# TP3 · Recomendador personalizado de libros

Procesamiento del Lenguaje Natural · TUIA · UNR-FCEIA · Grupo 7

Integrantes: Civetta Valentino, Fucci Milagros, Winter Federico, Frank Maximiliano. Lector de referencia: Valentino Civetta.

El notebook sigue los ocho pasos del enunciado con sus mismos títulos y está ejecutado completo en Google Colab con GPU. El paso 3 lo hizo solo el lector, en varias tandas y antes de ver cualquier predicción. Al final hay una sección de conclusiones.

| Paso | Qué hacemos |
|---|---|
| 1 | Cargar y revisar los libros |
| 2 | Auditar el catálogo (detectores de contaminación) |
| 3 | Registrar las preferencias del lector |
| 4 | Separar los libros para aprender y evaluar |
| 5 | Entrenar dos clasificadores (TF-IDF y embeddings) |
| 6 | Comprobar las predicciones (métricas y validación cruzada) |
| 7 | Convertir las predicciones en recomendaciones (precision@5) |
| 8 | Recomendar del catálogo completo, resumir y describir el perfil |

Archivos que genera dentro de `tp3/`: `grupo7_sospechosos.csv` y `data/catalogo_limpio.csv` (paso 2.7), y `grupo7_preferencias.csv` (pasos 3 y 4).

## Preparación

Además de pandas, scikit-learn y sentence-transformers (los del TP2, en `requirements.txt`), este TP usa `langdetect`, `jellyfish` (Jaro-Winkler) y `spacy` con el modelo `es_core_news_lg`. En Colab hay que descomentar y correr la celda de instalación una vez por sesión; si después pide reiniciar la sesión, se acepta y se sigue desde la celda siguiente.

El notebook se abre desde la carpeta `tp3/`. Los datos de entrada se leen de `tp3/data/` (variable `RUTA`) y los archivos de entrega se guardan en `tp3/` (variable `ENTREGA`). En Colab las dos tienen que apuntar a la carpeta donde se subieron los archivos, por ejemplo `/content/`.

In [ ]:
# Solo en Colab: descomentar y correr una vez por sesion (si pide reiniciar la sesion, aceptar)
# !pip install -q langdetect jellyfish spacy sentence-transformers
# !python -m spacy download es_core_news_lg

In [7]:
import re
import os
import collections
import unicodedata
import json

import numpy as np
import pandas as pd
import jellyfish
import spacy
from langdetect import detect_langs, DetectorFactory, LangDetectException
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score, f1_score, precision_recall_curve,
                             classification_report, confusion_matrix, make_scorer)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RepeatedStratifiedKFold, cross_val_score)
from sklearn.neighbors import NearestNeighbors

DetectorFactory.seed = 0      # langdetect es aleatorio: con la semilla fija repite el resultado
SEMILLA = 42
RUTA = "data/"      # datos de entrada (en Colab: "/content/")
ENTREGA = ""        # archivos de entrega: grupo7_sospechosos.csv y grupo7_preferencias.csv (en Colab: "/content/")

# Paso 1. Cargar y revisar los libros

Cargamos `catalogo_tp3.jsonl` y nos quedamos con `slug` (identificador), `title`, `authors` y `synopsis`. Vamos anotando cuántos libros sale cada limpieza para poder informarlo al final.

In [9]:
# leemos linea por linea con json: pd.read_json fallo con este archivo en la version de pandas de Colab
with open(RUTA + "catalogo_tp3.jsonl", encoding="utf-8") as f:
    catalogo = pd.DataFrame([json.loads(linea) for linea in f if linea.strip()])
libros = catalogo[["slug", "title", "authors", "synopsis"]].copy()

# authors viene como lista: lo pasamos a texto ordenado para poder comparar autores
libros["autores"] = libros["authors"].apply(
    lambda a: ", ".join(sorted(a)) if isinstance(a, list) else str(a))
libros = libros.drop(columns="authors")

eliminados = {}
total_inicial = len(libros)
print("Libros en el catalogo:", total_inicial)

Libros en el catalogo: 8747


## 1.1 Sin sinopsis y slugs repetidos

In [10]:
vacia = libros["synopsis"].isna() | (libros["synopsis"].astype(str).str.strip() == "")
eliminados["sin sinopsis"] = int(vacia.sum())
libros = libros[~vacia]

repetido = libros.duplicated("slug")
eliminados["slug repetido"] = int(repetido.sum())
libros = libros[~repetido]

print(eliminados)

{'sin sinopsis': 1, 'slug repetido': 0}


## 1.2 Normalizar espacios

Las palabras en cursiva quedaron separadas por saltos de línea. Reemplazamos cada secuencia de espacios en blanco por un solo espacio y no tocamos ningún otro carácter, porque en el paso 2 algunos caracteres raros (invisibles, de otro alfabeto) van a ser evidencia. `\s` no incluye el espacio de ancho cero (U+200B), así que esa evidencia se conserva.

In [11]:
con_salto = libros[libros["synopsis"].str.contains(r"\s{2,}|\n", regex=True)]
print("Sinopsis con saltos o espacios repetidos:", len(con_salto))
if len(con_salto):
    print("Antes:  ", repr(con_salto["synopsis"].iloc[0][:120]))

libros["synopsis"] = libros["synopsis"].apply(lambda t: re.sub(r"\s+", " ", t).strip())

if len(con_salto):
    print("Despues:", repr(libros.loc[con_salto.index[0], "synopsis"][:120]))

Sinopsis con saltos o espacios repetidos: 5253
Antes:   'La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más rápidos, panoramas inciertos.\nEud'
Despues: 'La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más rápidos, panoramas inciertos. Eud'


## 1.3 Sinopsis exactamente repetidas

Antes de borrar miramos qué hay detrás: cursos y series completas que repiten la misma sinopsis en todos sus volúmenes. Para esta práctica conservamos un libro por sinopsis (el primero que aparece).

In [12]:
grupos = libros[libros.duplicated("synopsis", keep=False)].groupby("synopsis")
tamanos = grupos.size().sort_values(ascending=False)
print("Sinopsis que se repiten:", len(tamanos), "| libros involucrados:", int(tamanos.sum()))

mayor = tamanos.index[0]
print("\nEl grupo mas grande tiene", tamanos.iloc[0], "libros, por ejemplo:")
print(libros[libros["synopsis"] == mayor][["title", "autores"]].head(5).to_string(index=False))

repetida = libros.duplicated("synopsis", keep="first")
eliminados["sinopsis repetida"] = int(repetida.sum())
libros = libros[~repetida]
print("\nEliminados por sinopsis repetida:", eliminados["sinopsis repetida"])

Sinopsis que se repiten: 33 | libros involucrados: 131

El grupo mas grande tiene 58 libros, por ejemplo:
             title        autores
Alemán – Unidad 40 Varios Autores
Alemán – Unidad 15 Varios Autores
Alemán – Unidad 13 Varios Autores
Alemán – Unidad 14 Varios Autores
Alemán – Unidad 34 Varios Autores

Eliminados por sinopsis repetida: 98


## 1.4 Versiones casi iguales de una misma obra

Comparamos solo libros del mismo autor, para no cruzar los 8.000 títulos entre sí. Jaro-Winkler (`jellyfish`) mide la forma de los títulos, no el significado, así que «Conversaciones con Dios I» y «Conversaciones con Dios II» salen casi idénticos aunque sean libros distintos.

El criterio combina título y sinopsis. Dos libros del mismo autor se unen si pasan estas tres condiciones:

1. Los títulos «clave» tienen Jaro-Winkler de 0,95 o más. La clave es el título en minúsculas, sin acentos, sin lo que va entre paréntesis (traductor, edición, ilustrado) y sin números de tomo o volumen.
2. Los números de tomo o volumen son los mismos (arábigos, romanos en mayúscula o «primero», «segundo»…). Así «Epigramas I» y «Epigramas II» quedan separados.
3. La clave es idéntica, o las sinopsis comparten el 95 % o más de sus palabras (Jaccard sobre conjuntos de palabras). Un título casi igual con una sinopsis distinta, como «Mitología oriental» y «Mitología occidental», es otro libro.

De cada par que se une conservamos el libro de sinopsis más larga.

In [13]:
UMBRAL_TITULO = 0.95
UMBRAL_SINOPSIS = 0.95
ORDINALES = r"\b(?:primero|segundo|tercero|cuarto|quinto)\b"

def sin_acentos(t):
    t = unicodedata.normalize("NFKD", t.lower())
    return "".join(c for c in t if not unicodedata.combining(c))

def numeros_titulo(t):
    """Numeros de tomo o volumen: arabigos, romanos en mayuscula y ordinales."""
    return frozenset(re.findall(r"\d+|\b[IVXLC]+\b", t) + re.findall(ORDINALES, sin_acentos(t)))

def clave_titulo(t):
    t = re.sub(r"\(.*?\)", "", t)                      # traductor, edicion, ilustrado
    t = re.sub(r"\d+|\b[IVXLC]+\b", "", t)             # numeros de tomo
    t = sin_acentos(t)
    t = re.sub(ORDINALES + r"|\b(?:vol|tomo|libro|parte)\b", "", t)
    t = re.sub(r"[^a-z ]", "", t)
    return re.sub(r"\s+", " ", t).strip()

def jaccard(a, b):
    return len(a & b) / len(a | b)

libros = libros.assign(
    clave=libros["title"].apply(clave_titulo),
    numeros=libros["title"].apply(numeros_titulo),
    palabras=libros["synopsis"].apply(lambda t: set(re.findall(r"\w+", t.lower()))),
    largo=libros["synopsis"].str.len(),
).sort_values("largo", ascending=False)

conservados = {}        # autores -> libros que ya decidimos conservar
descartar, pares_unidos, pares_separados = [], [], []

for r in libros.itertuples():
    unido = False
    for q in conservados.get(r.autores, []):
        if jellyfish.jaro_winkler_similarity(r.clave, q.clave) < UMBRAL_TITULO:
            continue
        jac = jaccard(r.palabras, q.palabras)
        if r.numeros != q.numeros:
            pares_separados.append((r.title, q.title, round(jac, 2), "tomo o volumen distinto"))
        elif r.clave == q.clave or jac >= UMBRAL_SINOPSIS:
            pares_unidos.append((r.title, q.title, round(jac, 2)))
            unido = True
            break
        else:
            pares_separados.append((r.title, q.title, round(jac, 2), "sinopsis distinta"))
    if unido:
        descartar.append(r.slug)
    else:
        conservados.setdefault(r.autores, []).append(r)

eliminados["casi duplicado"] = len(descartar)
libros = libros[~libros["slug"].isin(descartar)].sort_index()
libros = libros.drop(columns=["clave", "numeros", "palabras", "largo"]).reset_index(drop=True)

print("Eliminados por casi duplicado:", len(descartar))
print("\nDos pares que decidimos unir:")
display(pd.DataFrame(pares_unidos, columns=["titulo A", "titulo B (conservado)", "jaccard"]).head(2))
print("Dos pares que decidimos separar:")
display(pd.DataFrame(pares_separados, columns=["titulo A", "titulo B", "jaccard", "motivo"])
        .groupby("motivo").head(1))

Eliminados por casi duplicado: 55

Dos pares que decidimos unir:


,titulo A,titulo B (conservado),jaccard
0,La venganza de don Mendo (ed. Salvador García ...,La venganza de don Mendo,0.17
1,Cuentos completos (trad. Julio Cortázar),Cuentos completos (Penguin Clásicos),0.32


Dos pares que decidimos separar:


,titulo A,titulo B,jaccard,motivo
0,Gran escuela cartomágica Tomo II,Gran escuela cartomágica Tomo I,0.86,tomo o volumen distinto
4,Exploraciones secretas en Asia,Exploraciones secretas en África,0.17,sinopsis distinta


## 1.5 Resumen del paso 1

In [14]:
print("Libros al empezar:       ", total_inicial)
for motivo, n in eliminados.items():
    print(f"  eliminados por {motivo}: {n}")
print("Libros disponibles:      ", len(libros))
assert total_inicial - sum(eliminados.values()) == len(libros)

print("\nCinco registros al azar:")
display(libros.sample(5, random_state=SEMILLA)[["slug", "title", "autores"]])

Libros al empezar:        8747
  eliminados por sin sinopsis: 1
  eliminados por slug repetido: 0
  eliminados por sinopsis repetida: 98
  eliminados por casi duplicado: 55
Libros disponibles:       8593

Cinco registros al azar:


,slug,title,autores
4766,cuadernos-de-notas,Cuadernos de Notas,Leonardo da Vinci
6364,la-gran-pesquisa,La gran pesquisa,Tom Sharpe
1509,el-vicio-errante,El vicio errante,Jean Lorrain
7379,escultura-barroca-andaluza,Escultura barroca andaluza,Ernesto Ballesteros Arranz
6355,los-hijos-de-la-llanura-2,Los hijos de la llanura,Paula Vidal


# Paso 2. Auditar el catálogo

Trabajamos con `muestra_etiquetada.csv` (500 libros, 100 contaminados). La dividimos **por libro** en 70 % para desarrollar los detectores y 30 % para medirlos. Las decisiones (umbrales, patrones) salen siempre de desarrollo y las cifras finales se informan sobre prueba.

Hay que recordar que la muestra está enriquecida: 20 % contaminado contra una proporción mucho menor en el catálogo completo, así que la precisión medida acá es más optimista que la que tendríamos sobre los 8.000 libros. Además el catálogo tiene mensajes que la muestra nunca mostró.

In [15]:
muestra = pd.read_csv(RUTA + "muestra_etiquetada.csv", keep_default_na=False)

desarrollo, prueba = train_test_split(
    muestra, test_size=0.3, random_state=SEMILLA, stratify=muestra["contaminado"])
print(len(desarrollo), len(prueba))
print("contaminados en desarrollo:", int(desarrollo["contaminado"].sum()),
      "| en prueba:", int(prueba["contaminado"].sum()))

def medir(nombre, marcado, datos):
    y = datos["contaminado"].values
    marcado = np.asarray(marcado, dtype=bool)
    p = precision_score(y, marcado, zero_division=0)
    r = recall_score(y, marcado)
    print(f"{nombre}: precision {p:.2f}, recall {r:.2f}, marcados {int(marcado.sum())}")
    return p, r

def oraciones(texto):
    partes = re.split(r"(?<=[.!?])\s+|\n", texto)
    return [p.strip() for p in partes if len(p.strip()) >= 15]

# cada detector recibe un DataFrame con la columna synopsis y devuelve un arreglo de True/False
DETECTORES = {}

350 150
contaminados en desarrollo: 70 | en prueba: 30


## 2.1 Expresiones regulares

Una expresión para URLs (con `http(s)://` o `www.`), otra para correos y otra para teléfonos con prefijo internacional. Las unimos en una sola.

In [16]:
URL = r"https?://\S+|www\.\S+"
MAIL = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
TEL = r"\+\d[\d\s-]{8,}\d"
RE_BASE = re.compile("|".join([URL, MAIL, TEL]), re.IGNORECASE)

def det_regex(df):
    return df["synopsis"].apply(lambda t: bool(RE_BASE.search(t))).values

DETECTORES["regex"] = det_regex

marcado = det_regex(desarrollo)
medir("regex base (desarrollo)", marcado, desarrollo)

print("\nFalsos positivos (sinopsis legitimas marcadas):")
for t in desarrollo[marcado & (desarrollo["contaminado"] == 0)]["synopsis"]:
    print("  ", [m.group() for m in RE_BASE.finditer(t)])

perdidos = desarrollo[~marcado & (desarrollo["contaminado"] == 1)]
print(f"\nContaminados que la regex no ve: {len(perdidos)}. Algunos fragmentos:")
for f in perdidos["fragmento"].sample(min(8, len(perdidos)), random_state=SEMILLA):
    print("  ", ascii(f)[:110])

regex base (desarrollo): precision 0.94, recall 0.24, marcados 18

Falsos positivos (sinopsis legitimas marcadas):
   ['http://www.xkcd.com']

Contaminados que la regex no ve: 53. Algunos fragmentos:
   'Entr\xe1 a \u0441\u0430p\u0456tulol\u0456br\u0435.invalid y le\xe9 sin l\xedmites.'
   'Si te gust\xf3 esta rese\xf1a, compartila con tus amigos y particip\xe1 del sorteo de un lector electr\xf3nic
   'Una historia apasionante que te atrapar\xe1 desde la primera p\xe1gina hasta el final. Visit\xe1 bibli\u2060o
   'Link en nuestro sitio: h t t p s : / / epubhub [.] example'
   'Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!'
   'Si te gust\xf3 esta rese\xf1a, compartila con tus amigos y particip\xe1 del sorteo de un lector electr\xf3nic
   'Una obra imprescindible que no puede faltar en tu biblioteca personal. Visit\xe1 rela\u200btohub.test para la
   'Si te gust\xf3 esta rese\xf1a, compartila con tus amigos y particip\xe1 del sorteo de un lector electr\xf

**Qué detecta bien y qué no.** La regex acierta con los enlaces escritos de forma explícita (`http://…`, `www.…`), los correos y los teléfonos con `+`. No ve los enlaces camuflados ni el texto promocional sin enlace. Los falsos positivos aparecen cuando una sinopsis real menciona un sitio que existe (en la muestra hay una que nombra `xkcd.com`), y una regex no sabe distinguir una mención legítima de un enlace insertado. Cada falso positivo es un libro legítimo que después no se puede recomendar.

## 2.2 Normalizar antes de buscar

Los enlaces disimulados usan palabras en lugar de signos («punto», «barra»), letras separadas por espacios, `[.]`, letras de otro alfabeto que se ven iguales a las latinas y caracteres invisibles. `normalizar(texto)` deshace cada truco y después volvemos a aplicar la regex.

Con `unicodedata` se puede ver qué es cada carácter: una `o` latina, una `о` cirílica y un espacio de ancho cero se ven casi igual (o directamente no se ven) pero tienen nombres y categorías distintas.

In [17]:
for c in ("o", "о", "​", "⁠"):
    print(repr(c), unicodedata.name(c), unicodedata.category(c))

def con_invisibles(t):
    return any(unicodedata.category(c) == "Cf" for c in t)       # Cf = formato (invisibles)

inv = desarrollo["synopsis"].apply(con_invisibles)
print("\nSinopsis con caracteres invisibles -> contaminadas:",
      int((inv & (desarrollo["contaminado"] == 1)).sum()),
      "| legitimas:", int((inv & (desarrollo["contaminado"] == 0)).sum()))

'o' LATIN SMALL LETTER O Ll
'о' CYRILLIC SMALL LETTER O Ll
'\u200b' ZERO WIDTH SPACE Cf
'\u2060' WORD JOINER Cf

Sinopsis con caracteres invisibles -> contaminadas: 7 | legitimas: 4


Los invisibles también aparecen en sinopsis legítimas (guiones blandos, por ejemplo). Por sí solos no son evidencia: los quitamos para poder ver el texto que hay alrededor, no para marcar el libro.

In [18]:
TLD = r"com|net|org|info|io|ar|example|test|invalid"
DOMINIO = r"\b[\w-]+(?:\.[\w-]+)*\.(?:%s)\b" % TLD
RE_AMPLIADA = re.compile("|".join([URL, MAIL, TEL, DOMINIO]), re.IGNORECASE)

# letras cirilicas y griegas que se ven como latinas -> su equivalente latino
HOMOGLIFOS = str.maketrans(
    "аеорсіухοАЕОРСІХ",
    "aeopciyxoAEOPCIX")

def normalizar(texto):
    t = unicodedata.normalize("NFKC", texto)                       # formas de ancho completo, ligaduras
    t = "".join(c for c in t if unicodedata.category(c) != "Cf")   # invisibles
    t = t.translate(HOMOGLIFOS)                                    # otro alfabeto
    t = re.sub(r"\s*(?:\[\.\]|\(\.\))\s*", ".", t)                 # tomoya [.] test
    t = re.sub(r"(?<!\w)(?:\S ){3,}\S(?!\w)",                      # h t t p s : / /  y  w w w .
               lambda m: m.group().replace(" ", ""), t)
    t = re.sub(r"(?i)\s+punto\s+(%s)\b(\s+barra\s+)?" % TLD,       # sitio punto test barra libro
               lambda m: "." + m.group(1) + ("/" if m.group(2) else ""), t)
    t = re.sub(r"(\w) \. (\w)", r"\1.\2", t)                       # novelalibre . example
    t = re.sub(r"(?i)\b(https?://|www\.)\s+", r"\1", t)
    return t

det_base_norm = lambda df: df["synopsis"].apply(lambda t: bool(RE_BASE.search(normalizar(t)))).values
det_norm = lambda df: df["synopsis"].apply(lambda t: bool(RE_AMPLIADA.search(normalizar(t)))).values
DETECTORES["normalizacion"] = det_norm

medir("regex base sobre texto crudo      ", det_regex(desarrollo), desarrollo)
medir("regex base sobre texto normalizado", det_base_norm(desarrollo), desarrollo)
medir("regex + dominios, normalizado     ", det_norm(desarrollo), desarrollo)

print("\nDos casos que la normalizacion recupero (fragmento original -> normalizado):")
rec = [f for f in desarrollo[desarrollo["contaminado"] == 1]["fragmento"]
       if not RE_BASE.search(f) and RE_AMPLIADA.search(normalizar(f)) and normalizar(f) != f]
for f in rec[:2]:
    print("  ", ascii(f)[:100]); print("  ->", normalizar(f)[:100])

regex base sobre texto crudo      : precision 0.94, recall 0.24, marcados 18
regex base sobre texto normalizado: precision 0.96, recall 0.33, marcados 24
regex + dominios, normalizado     : precision 0.98, recall 0.60, marcados 43

Dos casos que la normalizacion recupero (fragmento original -> normalizado):
   'Una historia apasionante que te atrapar\xe1 desde la primera p\xe1gina hasta el final. Visit\xe1 pa
  -> Una historia apasionante que te atrapará desde la primera página hasta el final. Visitá paginazona.i
   'Entr\xe1 a libr\u043eexpress.invalid y le\xe9 sin l\xedmites.'
  -> Entrá a libroexpress.invalid y leé sin límites.


La segunda regex agrega una sola cosa: los dominios sin `http` ni `www` (`algo.example`, `algo.test`, `algo.com`). Los falsos positivos de esta versión siguen siendo las menciones legítimas de sitios reales; se cuentan en la columna `falsos_positivos` de la tabla del paso 2.7.

## 2.3 Detectar el idioma de cada oración

Primero el idioma de la sinopsis completa con `langdetect`. Esperamos que un mensaje corto en inglés dentro de un texto largo en español no alcance a cambiar el resultado; la tabla lo confirma o lo desmiente.

In [19]:
def idioma_completo(texto):
    try:
        return detect_langs(texto)[0].lang
    except LangDetectException:
        return "?"

des_idioma = desarrollo.assign(idioma=desarrollo["synopsis"].apply(idioma_completo))
display(pd.crosstab(des_idioma["contaminado"], des_idioma["idioma"]))

idioma,es
contaminado,
0,280
1,70


Dividimos cada sinopsis en oraciones y detectamos el idioma de cada una. Las oraciones cortas, las listas de nombres propios y las citas producen errores, así que hay que fijar dos valores: la cantidad mínima de palabras y la probabilidad mínima para creer que la oración **no** está en español. Probamos una grilla sobre desarrollo y elegimos la combinación de mayor recall que mantenga una precisión de al menos 0,90.

In [20]:
def idiomas_oraciones(texto):
    salida = []
    for o in oraciones(texto):
        try:
            l = detect_langs(o)[0]
            salida.append((len(o.split()), l.lang, l.prob))
        except LangDetectException:
            pass
    return salida

desarrollo = desarrollo.assign(idiomas=desarrollo["synopsis"].apply(idiomas_oraciones))

def marca_idioma(idiomas, min_palabras, umbral):
    return any(n >= min_palabras and lang != "es" and p >= umbral for n, lang, p in idiomas)

filas = []
for mp in (2, 3, 4, 5, 6):
    for um in (0.7, 0.8, 0.9, 0.95, 0.99):
        pred = desarrollo["idiomas"].apply(lambda x: marca_idioma(x, mp, um))
        y = desarrollo["contaminado"].values
        filas.append(dict(min_palabras=mp, umbral=um,
                          precision=precision_score(y, pred, zero_division=0),
                          recall=recall_score(y, pred)))
grilla = pd.DataFrame(filas)
display(grilla.pivot(index="min_palabras", columns="umbral", values="precision").round(2))
display(grilla.pivot(index="min_palabras", columns="umbral", values="recall").round(2))

elegibles = grilla[grilla["precision"] >= 0.90]
if elegibles.empty:
    print("Ninguna combinacion llega a precision 0.90: usamos la de mejor precision")
    elegibles = grilla.sort_values("precision", ascending=False).head(1)
# ante empate de recall nos quedamos con la combinacion mas exigente (umbral y minimo mas altos)
mejor = elegibles.sort_values(["recall", "umbral", "min_palabras"], ascending=False).iloc[0]
MIN_PALABRAS, UMBRAL_IDIOMA = int(mejor["min_palabras"]), float(mejor["umbral"])
print(f"\nElegimos min_palabras={MIN_PALABRAS} y umbral={UMBRAL_IDIOMA} "
      f"(desarrollo: precision {mejor['precision']:.2f}, recall {mejor['recall']:.2f})")

def det_idioma(df):
    return df["synopsis"].apply(
        lambda t: marca_idioma(idiomas_oraciones(t), MIN_PALABRAS, UMBRAL_IDIOMA)).values

DETECTORES["idioma"] = det_idioma

umbral,0.70,0.80,0.90,0.95,0.99
min_palabras,,,,,
2,0.37,0.38,0.43,0.43,0.43
3,0.39,0.40,0.44,0.44,0.44
4,0.45,0.46,0.55,0.55,0.55
5,0.53,0.55,0.57,0.57,0.57
6,0.62,0.65,0.70,0.70,0.70


umbral,0.70,0.80,0.90,0.95,0.99
min_palabras,,,,,
2,0.31,0.30,0.29,0.29,0.29
3,0.31,0.30,0.29,0.29,0.29
4,0.27,0.26,0.26,0.26,0.26
5,0.26,0.24,0.24,0.24,0.24
6,0.23,0.21,0.20,0.20,0.20


Ninguna combinacion llega a precision 0.90: usamos la de mejor precision

Elegimos min_palabras=6 y umbral=0.99 (desarrollo: precision 0.70, recall 0.20)


**Por qué elegimos 6 palabras y 0,99.** La tabla de arriba muestra que el idioma de la sinopsis completa no sirve: langdetect dice `es` para los 350 libros de desarrollo, contaminados o no, porque un mensaje de una oración no alcanza a cambiar el idioma de un texto de casi mil caracteres.

Por oración sí aparece la señal, pero con ruido. En la grilla, subir el mínimo de palabras es lo que más mejora la precisión (con umbral 0,9 pasa de 0,43 con 2 palabras a 0,70 con 6) y a la vez baja el recall (de 0,29 a 0,20). Las oraciones cortas, las listas de nombres propios y los títulos o citas en otro idioma son las que langdetect confunde, y un mínimo alto las deja afuera. El umbral casi no cambia nada desde 0,9 en adelante, porque langdetect da probabilidades muy altas aun cuando se equivoca. Ninguna combinación llegó a la precisión de 0,90 que buscábamos, así que nos quedamos con la de mayor precisión (6 palabras y 0,99), sabiendo que es el detector más flojo.

## 2.4 Reconocer entidades

Aplicamos NER con spaCy (`es_core_news_lg`) a las oraciones de los fragmentos contaminados de desarrollo y a la misma cantidad de oraciones de sinopsis limpias.

In [21]:
nlp = spacy.load("es_core_news_lg")

frases_malas = [o for f in desarrollo[desarrollo["contaminado"] == 1]["fragmento"] for o in oraciones(f)]
limpias = desarrollo[desarrollo["contaminado"] == 0]["synopsis"].apply(oraciones).explode().dropna()
frases_buenas = limpias.sample(len(frases_malas), random_state=SEMILLA).tolist()

def entidades(frases):
    cuenta, textos = collections.Counter(), collections.Counter()
    for doc in nlp.pipe(frases, batch_size=64):
        for e in doc.ents:
            cuenta[e.label_] += 1
            textos[(e.label_, e.text)] += 1
    return cuenta, textos

c_mal, t_mal = entidades(frases_malas)
c_bien, t_bien = entidades(frases_buenas)

print(len(frases_malas), "oraciones de cada grupo")
display(pd.DataFrame({"contaminadas": c_mal, "limpias": c_bien}).fillna(0).astype(int))
print("Entidades mas frecuentes en el texto insertado:", t_mal.most_common(8))
print("Entidades mas frecuentes en sinopsis limpias:  ", t_bien.most_common(8))

88 oraciones de cada grupo


,contaminadas,limpias
MISC,37,14
LOC,15,23
PER,16,30
ORG,2,7


Entidades mas frecuentes en el texto insertado: [(('PER', 'compartila'), 7), (('MISC', 'PDF'), 6), (('PER', 'Ganá'), 5), (('MISC', 'Download this ebook for FREE'), 4), (('MISC', 'sign-up required'), 4), (('MISC', 'EPUB sin registro'), 4), (('MISC', 'Descargalo'), 4), (('LOC', 'Oferta'), 2)]
Entidades mas frecuentes en sinopsis limpias:   [(('LOC', 'Europa'), 4), (('LOC', 'Ellis'), 2), (('LOC', 'la India'), 2), (('PER', 'Hitler'), 2), (('PER', 'Platero'), 2), (('PER', 'Dispenza'), 1), (('PER', 'Goethe'), 1), (('MISC', 'Tiempos de barro'), 1)]


**Qué encontró el NER.** En el texto insertado predomina la etiqueta MISC (37 contra 14 en las oraciones limpias), mientras que en las sinopsis normales hay más personas y lugares (PER 30 y LOC 23, contra 16 y 15). Las MISC del texto insertado son pedazos de la publicidad: «Download this ebook for FREE», «sign-up required», «EPUB sin registro», «PDF». También aparecen verbos en imperativo tomados como personas («compartila», «Ganá»). spaCy no tiene etiquetas para sitio web, precio o marca, así que mete todo eso en MISC o directamente se equivoca.

Por eso el NER sirve para explicar lo que encontraron los otros detectores (formatos de archivo, ofertas, llamados a la acción), y no lo usamos para detectar. Una sinopsis legítima también tiene MISC, por ejemplo títulos de obras como «Tiempos de barro», y con estos errores de etiqueta marcaría muchos libros limpios.

## 2.5 Clasificar oraciones

El texto promocional sin enlaces solo se detecta aprendiendo de ejemplos. Armamos un conjunto de oraciones con los libros de desarrollo: las oraciones del fragmento son positivas y las oraciones de las sinopsis limpias son negativas. Entrenamos dos clasificadores como los del paso 5: uno con TF-IDF y otro con embeddings (`distiluse-base-multilingual-cased-v1`, el mismo modelo del TP2), los dos con `LogisticRegression(class_weight="balanced", max_iter=1000)`.

Un libro es sospechoso si alguna de sus oraciones supera el umbral. Elegimos ese umbral con `precision_recall_curve` para una precisión de 0,90, no con 0,5. Para que las probabilidades de desarrollo no sean optimistas (el modelo vería los mismos libros que se están midiendo) las calculamos con validación cruzada **por libro**: cada libro se puntúa con un modelo que no lo vio entrenar.

In [22]:
modelo_emb = SentenceTransformer("distiluse-base-multilingual-cased-v1")

def armar_oraciones(df):
    filas = []
    for r in df.itertuples():
        textos = oraciones(r.fragmento) if r.contaminado == 1 else oraciones(r.synopsis)
        filas += [(r.slug, o, int(r.contaminado)) for o in textos]
    return pd.DataFrame(filas, columns=["slug", "oracion", "y"])

def ajustar(tipo, textos, y):
    """Entrena la regresion logistica y devuelve una funcion textos -> probabilidad de contaminado."""
    textos = list(textos)
    if tipo == "tfidf":
        vec = TfidfVectorizer()
        X = vec.fit_transform(textos)
        codificar = vec.transform
    else:
        codificar = lambda t: modelo_emb.encode(list(t), show_progress_bar=False)
        X = codificar(textos)
    clf = LogisticRegression(class_weight="balanced", max_iter=1000).fit(X, y)
    assert list(clf.classes_) == [0, 1]      # la columna 1 de predict_proba es "contaminado"
    return lambda t: clf.predict_proba(codificar(t))[:, 1]

def puntajes_libros(f_prob, df):
    """Puntaje de cada libro = mayor probabilidad entre sus oraciones."""
    por_libro = df["synopsis"].apply(oraciones).tolist()
    planas = [(i, o) for i, lista in enumerate(por_libro) for o in lista]
    puntaje = np.zeros(len(df))
    if planas:
        probs = f_prob([o for _, o in planas])
        for (i, _), p in zip(planas, probs):
            puntaje[i] = max(puntaje[i], p)
    return puntaje

def puntajes_cruzados(tipo, df, k=5):
    puntaje = np.zeros(len(df))
    for ent, val in StratifiedKFold(k, shuffle=True, random_state=SEMILLA).split(df, df["contaminado"]):
        o = armar_oraciones(df.iloc[ent])
        f = ajustar(tipo, o["oracion"], o["y"])
        puntaje[val] = puntajes_libros(f, df.iloc[val])
    return puntaje

def elegir_umbral(y, puntaje, precision_min=0.90):
    p, r, u = precision_recall_curve(y, puntaje)
    ok = np.where(p[:-1] >= precision_min)[0]
    if len(ok) == 0:
        print("Ninguno llega a la precision pedida: usamos el umbral de mayor precision")
        return u[np.argmax(p[:-1])]
    return u[ok[np.argmax(r[ok])]]

oraciones_des = armar_oraciones(desarrollo)
print("Oraciones de entrenamiento:", len(oraciones_des),
      "| positivas:", int(oraciones_des["y"].sum()))

y_des = desarrollo["contaminado"].values
clasificadores = {}
for tipo in ("tfidf", "emb"):
    puntaje_cv = puntajes_cruzados(tipo, desarrollo)
    umbral = elegir_umbral(y_des, puntaje_cv)
    f_prob = ajustar(tipo, oraciones_des["oracion"], oraciones_des["y"])    # modelo final, todo desarrollo
    clasificadores[tipo] = (f_prob, umbral)
    print(f"{tipo}: umbral elegido {umbral:.3f}")
    medir(f"{tipo} (desarrollo, validacion cruzada)", puntaje_cv >= umbral, desarrollo)

def hacer_detector(tipo):
    f_prob, umbral = clasificadores[tipo]
    return lambda df: puntajes_libros(f_prob, df) >= umbral

DETECTORES["clasificador_tfidf"] = hacer_detector("tfidf")
DETECTORES["clasificador_emb"] = hacer_detector("emb")

for nombre in ("clasificador_tfidf", "clasificador_emb"):
    medir(nombre + " (prueba)", DETECTORES[nombre](prueba), prueba)

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/556 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 1.58MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

Oraciones de entrenamiento: 2095 | positivas: 88
tfidf: umbral elegido 0.441
tfidf (desarrollo, validacion cruzada): precision 0.90, recall 0.91, marcados 71
emb: umbral elegido 0.567
emb (desarrollo, validacion cruzada): precision 0.90, recall 0.93, marcados 72
clasificador_tfidf (prueba): precision 1.00, recall 0.93, marcados 28
clasificador_emb (prueba): precision 0.90, recall 0.87, marcados 29


**Respuestas.**

1. Conviene clasificar oraciones porque el texto insertado es corto: en la muestra mide 81 caracteres en promedio, y una sinopsis del catálogo unos 970. Si se representa la sinopsis entera, la frase promocional queda diluida en el resto del vector y el modelo aprende sobre todo de qué trata el libro. Por oración el ejemplo positivo es casi puro, y alcanza con que una sola oración pase el umbral para marcar el libro.
2. Con el umbral elegido para precisión 0,90 en desarrollo, los dos quedaron casi empatados en validación cruzada (recall 0,91 TF-IDF y 0,93 embeddings). En prueba ganó TF-IDF: precisión 1,00 y recall 0,93, con 28 libros marcados y ninguno limpio, contra 0,90 y 0,87 de embeddings (29 marcados, 3 limpios). A TF-IDF se le escaparon dos libros: uno con un correo (`ventas@tomozona.test`) y otro con un enlace de letras separadas (`h t t p s : / / lectortotal [.] example`). Esos dos los encuentran igual la regex y la normalización.
3. A TF-IDF le va bien porque la publicidad de la muestra repite palabras exactas («Visitá», «gratis», `.example`, «sorteo») y con 2.095 oraciones de entrenamiento las aprende rápido. También es mejor para cadenas literales como una URL. Pero solo reconoce palabras que ya vio. Para los mensajes nuevos del catálogo, con el mismo sentido pero otras palabras u otro idioma, esperamos que generalicen mejor los embeddings, porque el modelo es multilingüe y compara significados. Por eso en el paso 2.7 usamos los dos.

## 2.6 Encontrar libros copiados

Algunos libros no existen: copian la sinopsis de otro libro del catálogo y le agregan texto. Para cada libro buscamos su vecino más parecido con similitud del coseno (`NearestNeighbors`, `metric="cosine"`, sobre TF-IDF) en el catálogo que quedó del paso 1.

Una pareja con similitud alta y **autores distintos** es candidata, porque las series legítimas también comparten texto pero casi siempre tienen el mismo autor. Para decidir cuál de los dos es la copia usamos que la copia contiene a la otra sinopsis completa más texto agregado; la copia es siempre la sinopsis **más larga** de la pareja. Jaccard sobre palabras confirma el parecido. Si el texto agregado quedó en el medio y la copia ya no contiene a la otra de corrido, igual la marcamos cuando la similitud es de 0,90 o más y el Jaccard está entre 0,80 y 1 (idéntico no es copia, es el mismo texto repetido).

In [23]:
SIM_MIN = 0.70
JACCARD_MIN = 0.50

tfidf_nn = TfidfVectorizer()
X_nn = tfidf_nn.fit_transform(libros["synopsis"])
nn = NearestNeighbors(n_neighbors=3, metric="cosine", algorithm="brute").fit(X_nn)
dist, vecinos = nn.kneighbors(X_nn)

pares = {}
for i in range(len(libros)):
    for d, j in zip(dist[i], vecinos[i]):
        if j != i:                                  # el primer vecino distinto de si mismo
            pares[tuple(sorted((i, j)))] = 1 - d
            break

filas, copias = [], set()
for (i, j), sim in pares.items():
    a, b = libros.iloc[i], libros.iloc[j]
    if sim < SIM_MIN or a["autores"] == b["autores"]:
        continue
    jac = jaccard(set(re.findall(r"\w+", a["synopsis"].lower())),
                  set(re.findall(r"\w+", b["synopsis"].lower())))
    corta, larga = (a, b) if len(a["synopsis"]) <= len(b["synopsis"]) else (b, a)
    contiene = corta["synopsis"] in larga["synopsis"]
    # la copia puede haber insertado el texto en el medio, y entonces ya no contiene a la otra sinopsis
    # de corrido: si el parecido es muy alto (pero no identico) tambien la contamos
    parecida = sim >= 0.90 and 0.80 <= jac < 1.0
    if (contiene and jac >= JACCARD_MIN) or parecida:
        copias.add(larga["slug"])
    filas.append(dict(copia_probable=larga["title"], autor_copia=larga["autores"],
                      original=corta["title"], autor_original=corta["autores"],
                      similitud=round(sim, 2), jaccard=round(jac, 2), contiene=contiene))

candidatos = pd.DataFrame(filas)
print("Parejas con similitud >=", SIM_MIN, "y autores distintos:", len(candidatos))
print("Libros marcados como copia:", len(copias))
if len(candidatos):
    display(candidatos.sort_values("similitud", ascending=False).head(10))
    print("Parejas que quedaron para revisar a mano (no se contienen):")
    display(candidatos[~candidatos["contiene"]].head(5))

DETECTORES["copia"] = lambda df: df["slug"].isin(copias).values
medir("copia (desarrollo)", DETECTORES["copia"](desarrollo), desarrollo)
medir("copia (prueba)", DETECTORES["copia"](prueba), prueba)

Parejas con similitud >= 0.7 y autores distintos: 77
Libros marcados como copia: 61


,copia_probable,autor_copia,original,autor_original,similitud,jaccard,contiene
30,Las grandes preguntas de los niños y las senci...,Gemma Elwin Harris,Las grandes preguntas de los niños y las senci...,Varios Autores,1.00,1.00,False
37,La última tormenta,Lucía Oyarzún,La pequeña comunista que no sonreía nunca,Lola Lafon,0.99,0.94,False
64,Los hijos de la promesa,Paula Arrieta,Elvira de Hidalgo,Juan Villalba Sebastián,0.99,0.95,True
7,Crónica del bosque,Sofía Correa,Cotufas en el golfo,Gonzalo Torrente Ballester,0.98,0.92,True
5,Richard P. Feynman. La física de las palabras,Michelle Feynman,La física de las palabras,Richard Phillips Feynman,0.98,0.93,True
66,La última herencia,Andrés Barrionuevo,En frío,Bjørn Lomborg,0.98,0.94,True
73,Bajo la isla,Clara Correa,Trás-os-Montes,Julio Llamazares,0.98,0.95,True
36,Los hijos de la marea,Martín Saravia,El último lector,Ricardo Piglia,0.97,0.89,True
17,Memorias del puerto,Andrés Peralta,Dormir sin lágrimas,Rosa Jové,0.97,0.94,True
27,Memorias del faro,Tomás Barrionuevo,Las infinitas vidas de Euclides : historia del...,Benjamin Wardhaugh,0.97,0.92,True


Parejas que quedaron para revisar a mano (no se contienen):


,copia_probable,autor_copia,original,autor_original,similitud,jaccard,contiene
0,Crónica del silencio,Martín Barrionuevo,La vida cotidiana en Pompeya,Robert Étienne,0.94,0.85,False
12,El camino de la llanura,Valeria Saravia,La Edad del Futuro I,Isaac Asimov,0.93,0.87,False
19,Torneo de terror,"H. William Stine, Megan Stine",¿Quién secuestró a la Princesa Saralinda?,"Bill Stine, Megan Stine",0.78,0.59,False
20,La amenaza del dueño del tiempo,Lynn Beach,¿Quién secuestró a la Princesa Saralinda?,"Bill Stine, Megan Stine",0.79,0.60,False
21,El desafío del caballero Lobo,R. L. Stine,¿Quién secuestró a la Princesa Saralinda?,"Bill Stine, Megan Stine",0.79,0.57,False


copia (desarrollo): precision 1.00, recall 0.10, marcados 7
copia (prueba): precision 1.00, recall 0.10, marcados 3


(1.0, 0.1)

Si hay parejas que no se contienen pero tienen similitud alta, hay que mirarlas a mano: el texto agregado puede estar en el medio o la copia puede haber retocado la sinopsis original. Las que están por debajo de 0,90 de similitud no se marcan automáticamente.

**Resultado.** En el catálogo marcamos 61 copias. En la muestra la precisión es 1,00 y el recall 0,10 (3 de 30 contaminados en prueba): las copias son un tipo de contaminación entre varios, y este detector no ve los demás. *Crónica del silencio* (Martín Barrionuevo) y *El camino de la llanura* (Valeria Saravia) repiten casi completa la sinopsis de otro libro sin contenerla de corrido, y la regla de 0,90 las marca igual. Las parejas de libros juveniles con *¿Quién secuestró a la Princesa Saralinda?* tienen similitud cercana a 0,78 y autores muy parecidos (Stine), así que quedan sin marcar.

## 2.7 Combinar y entregar

Un libro es sospechoso si al menos un detector lo marca. Marcar todo tampoco sirve: cada libro legítimo que se elimina es uno que nunca se va a poder recomendar, así que miramos la precisión y el recall de cada detector y de la unión sobre la parte de **prueba**.

In [24]:
y_pru = prueba["contaminado"].values
marcas_pru = {nombre: np.asarray(fn(prueba), dtype=bool) for nombre, fn in DETECTORES.items()}
marcas_pru["COMBINADO (al menos uno)"] = np.any(list(marcas_pru.values()), axis=0)

tabla = pd.DataFrame([
    dict(detector=n,
         precision=precision_score(y_pru, m, zero_division=0),
         recall=recall_score(y_pru, m),
         marcados=int(m.sum()),
         falsos_positivos=int((m & (y_pru == 0)).sum()))
    for n, m in marcas_pru.items()]).round(2)
display(tabla)

,detector,precision,recall,marcados,falsos_positivos
0,regex,1.00,0.23,7,0
1,normalizacion,1.00,0.63,19,0
2,idioma,0.62,0.33,16,6
3,clasificador_tfidf,1.00,0.93,28,0
4,clasificador_emb,0.90,0.87,29,3
5,copia,1.00,0.10,3,0
6,COMBINADO (al menos uno),0.79,1.00,38,8


Ahora aplicamos todos los detectores al catálogo completo del paso 1 y guardamos `grupo7_sospechosos.csv` en `tp3/` (columnas `slug` y `motivo`). Un libro que marcan varios detectores lleva varios motivos unidos con `+`. Si la regex ampliada solo marca lo que la regex base no veía, el motivo es `normalizacion`, y los dos clasificadores se agrupan en `clasificador`.

Con el catálogo de unos 8.500 libros el clasificador de embeddings codifica decenas de miles de oraciones: en Colab con GPU tarda unos minutos y con CPU bastante más.

In [25]:
marcas = {nombre: np.asarray(fn(libros), dtype=bool) for nombre, fn in DETECTORES.items()}

motivos_libro = {
    "regex": marcas["regex"],
    "normalizacion": marcas["normalizacion"] & ~marcas["regex"],
    "idioma": marcas["idioma"],
    "clasificador": marcas["clasificador_tfidf"] | marcas["clasificador_emb"],
    "copia": marcas["copia"],
}

def juntar_motivos(i):
    return "+".join(m for m, v in motivos_libro.items() if v[i])

sospechoso = np.any(list(motivos_libro.values()), axis=0)
sospechosos = pd.DataFrame({
    "slug": libros.loc[sospechoso, "slug"].values,
    "motivo": [juntar_motivos(i) for i in np.where(sospechoso)[0]]})

# si ya se corrio el paso 3, el lector agrego libros con motivo revision_manual: no los perdemos
guardar = sospechosos
if os.path.exists(ENTREGA + "grupo7_sospechosos.csv"):
    previos = pd.read_csv(ENTREGA + "grupo7_sospechosos.csv")
    manuales = previos[(previos["motivo"] == "revision_manual") & ~previos["slug"].isin(sospechosos["slug"])]
    guardar = pd.concat([sospechosos, manuales], ignore_index=True)
guardar.to_csv(ENTREGA + "grupo7_sospechosos.csv", index=False)

catalogo_limpio = libros[~sospechoso].reset_index(drop=True)
catalogo_limpio.to_csv(RUTA + "catalogo_limpio.csv", index=False)

print("Libros revisados:", len(libros))
print("Sospechosos:", len(sospechosos), f"({len(sospechosos) / len(libros):.1%})")
print("Catalogo limpio:", len(catalogo_limpio))
display(sospechosos["motivo"].value_counts())

Libros revisados: 8593
Sospechosos: 703 (8.2%)
Catalogo limpio: 7890


,count
motivo,
clasificador,265
idioma,172
idioma+clasificador,65
regex+clasificador,50
normalizacion+clasificador,38
regex+clasificador+copia,19
clasificador+copia,19
normalizacion+idioma+clasificador,18
normalizacion,17


## Conclusión del paso 2

Sobre la parte de prueba, la combinación encontró los 30 libros contaminados (recall 1,00) con precisión 0,79: marcó 38 libros, 8 de ellos limpios. Seis de esos ocho falsos positivos vienen del detector de idioma, que solo tiene precisión 0,62. Cada detector aporta un tipo de contaminación: la regex, los enlaces explícitos; la normalización, los enlaces disimulados (letras separadas, `[.]`, cirílico, caracteres invisibles); el idioma, los mensajes en inglés o portugués; los clasificadores, la publicidad sin enlace; y los vecinos más cercanos, las copias. Los clasificadores son los que más libros encuentran.

En el catálogo completo marcamos 703 libros (8,2 %) y quedaron 7.890 en `catalogo_limpio.csv`. Es más de lo que esperábamos: la muestra tiene 20 % de contaminados y el enunciado aclara que el catálogo tiene bastante menos, así que una parte de esos 703 deben ser falsos positivos. El principal sospechoso es el detector de idioma, que marcó 261 libros, 172 de ellos sin que ningún otro detector coincidiera. Entre esos hay muchas antologías y revistas de ciencia ficción (*Nueva Dimensión*, *Más allá*, *Ciencia ficción. Selección*) que habría que revisar a mano.

Un falso positivo tiene un costo concreto: ese libro sale del catálogo y nunca se va a poder recomendar. Si lo hiciéramos de nuevo, pediríamos que el idioma coincida con otro detector antes de sacar el libro. No lo cambiamos ahora porque el lector ya había valorado los libros en el orden del catálogo limpio, y cambiar el catálogo cambia ese orden.

# Paso 3. Registrar las preferencias de un lector

**Esta parte la corre solo la persona del grupo que elegimos como lector de referencia**, en casa y antes de ver ninguna predicción. Elegido el lector: **Valentino Civetta**.

Mezclamos el catálogo limpio una sola vez con `df.sample(frac=1, random_state=42)`. Al lector se le muestra el título y la sinopsis de cada libro, en ese orden, y responde «¿Me interesaría leer este libro?»:

| Tecla | Respuesta | `interes` |
|---|---|---|
| `s` | Sí, me interesaría leerlo | 1 |
| `n` | No me interesaría leerlo | 0 |
| `v` | No puedo decidir | vacío |
| `m` | Parece contaminado y el paso 2 no lo detectó | vacío (y va a sospechosos con motivo `revision_manual`) |
| `q` | Corto acá y sigo después | no se guarda nada |

Se sigue hasta reunir al menos 100 respuestas válidas y 20 de cada clase. Cada respuesta se guarda en el archivo en el momento (igual que el scraper del TP1 escribe libro por libro), así que se puede cortar y retomar sin perder nada: al volver a correr la celda sigue desde el primer libro sin responder.

In [28]:
import textwrap

ARCH_PREF = ENTREGA + "grupo7_preferencias.csv"
ARCH_SOSP = ENTREGA + "grupo7_sospechosos.csv"
MINIMO_VALIDAS, MINIMO_POR_CLASE = 100, 20

limpio = pd.read_csv(RUTA + "catalogo_limpio.csv", keep_default_na=False)
orden = limpio.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)

if os.path.exists(ARCH_PREF):
    prefs = pd.read_csv(ARCH_PREF, keep_default_na=False)       # el interes vacio queda como ""
else:
    prefs = pd.DataFrame(columns=["slug", "title", "synopsis", "interes"])

def conteo(p):
    v = pd.to_numeric(p["interes"], errors="coerce")
    return int((v == 1).sum()), int((v == 0).sum())

def completo(p):
    pos, neg = conteo(p)
    return pos + neg >= MINIMO_VALIDAS and pos >= MINIMO_POR_CLASE and neg >= MINIMO_POR_CLASE

def marcar_revision_manual(slug):
    sos = pd.read_csv(ARCH_SOSP) if os.path.exists(ARCH_SOSP) else pd.DataFrame(columns=["slug", "motivo"])
    if slug not in set(sos["slug"]):
        sos.loc[len(sos)] = [slug, "revision_manual"]
        sos.to_csv(ARCH_SOSP, index=False)

RESPUESTAS = {"s": 1, "n": 0, "v": "", "m": ""}
vistos = set(prefs["slug"])

for libro in orden.itertuples():
    if completo(prefs):
        break
    if libro.slug in vistos:
        continue
    pos, neg = conteo(prefs)
    print("\n" + "=" * 70)
    print(f"Respuestas validas: {pos + neg}  (si: {pos}, no: {neg})   meta: {MINIMO_VALIDAS} y {MINIMO_POR_CLASE} de cada clase\n")
    print(libro.title, "\n")
    print(textwrap.fill(libro.synopsis, 100), "\n")
    r = ""
    while r not in RESPUESTAS and r != "q":
        r = input("Te interesaria leerlo? [s / n / v = no puedo decidir / m = contaminado / q = salir]: ").strip().lower()
    if r == "q":
        break
    if r == "m":
        marcar_revision_manual(libro.slug)
    prefs.loc[len(prefs), ["slug", "title", "synopsis", "interes"]] = [libro.slug, libro.title, libro.synopsis, RESPUESTAS[r]]
    prefs.to_csv(ARCH_PREF, index=False)
    vistos.add(libro.slug)

print("Listo." if completo(prefs) else "Quedaron respuestas pendientes: volver a correr esta celda.")


Respuestas validas: 2  (si: 2, no: 0)   meta: 100 y 20 de cada clase

Diccionario de la ciencia 

Sabemos que la ciencia penetra nuestras vidas, que las condiciona cada vez más profunda e
intensamente. Y, sin embargo, para la mayoría de nosotros, el conocimiento científico es algo ajeno
que contemplamos con una mezcla —surgida habitualmente de la ignorancia— de respeto y temor. Este
diccionario pretende acercar esa aparentemente todopoderosa ciencia a todo tipo de lectores,
incluyendo entre ellos a los propios científicos, alejados en su mayoría, en esta era de la
especialización y compartimentación, de una visión global —y humana— de su disciplina, esencialmente
múltiple, por otra parte. Pero para lograr semejante compendio —cuanto más completo y escueto mejor—
que permita entender el mayor número posible de aportaciones científicas. Lo que este diccionario
presenta es una visión personal, profundamente idiosincrásica y selectiva, apasionada e intensa de
la ciencia. Una visión en la 

/tmp/ipykernel_4823/2580076160.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  prefs.loc[len(prefs), ["slug", "title", "synopsis", "interes"]] = [libro.slug, libro.title, libro.synopsis, RESPUESTAS[r]]



Respuestas validas: 11  (si: 5, no: 6)   meta: 100 y 20 de cada clase

El laberinto del Fauno 

Guión de la película del realizador mexicano Guillermo del Toro. 

Te interesaria leerlo? [s / n / v = no puedo decidir / m = contaminado / q = salir]: n

Respuestas validas: 12  (si: 5, no: 7)   meta: 100 y 20 de cada clase

Carpe diem 

Carpe diem. Vive el momento. Sumérgete en el aquí y el ahora. Esas frases salidas de la boca del
extravagante doctor Tamkin como un dudoso elixir milagroso rebotan contra los tímpanos de Wilhelm,
un hombre acosado por varios frentes: actor fracasado, danza al son de las exigencias de su ex mujer
y sus dos hijos, despreciado por la empresa que le despidió en lugar de otorgarle el ascenso
prometido y ninguneado por la soberbia y frialdad de su propio padre. Pero la esperanza es lo último
que se pierde, y ahora ésta se ha encarnado en setecientos dólares... La narración envolvente de
Carpe diem, su mordaz sentido del humor, la descripción minuciosa de la geog

## Resumen de las preferencias

Un libro sin valorar tiene preferencia desconocida: queda con `interes` vacío y no se convierte en 0.

In [29]:
prefs = pd.read_csv(ARCH_PREF)               # aca el interes vacio pasa a NaN
print("Libros vistos:", len(prefs))
print(prefs["interes"].map({1: "interes=1", 0: "interes=0"}).fillna("sin valorar").value_counts())

pos, neg = conteo(prefs.fillna(""))
print(f"\nRespuestas validas: {pos + neg} (interes=1: {pos}, interes=0: {neg})")
assert completo(prefs.fillna("")), "Todavia faltan respuestas para llegar a 100 validas y 20 de cada clase"

Libros vistos: 101
interes
interes=0      54
interes=1      46
sin valorar     1
Name: count, dtype: int64

Respuestas validas: 100 (interes=1: 46, interes=0: 54)


El lector vio 101 libros en el orden mezclado: 46 le interesaron, 54 no y uno quedó sin valorar porque no pudo decidir (*Lacan*). Con eso llegó a las 100 respuestas válidas y a más de 20 de cada clase. No encontró ningún libro contaminado que se le hubiera escapado al paso 2, por eso `grupo7_sospechosos.csv` no tiene filas con motivo `revision_manual`.

# Paso 4. Separar los libros para aprender y evaluar

Trabajamos solo con los libros que tienen `interes` igual a 0 o 1. Los que quedaron en blanco (el lector no pudo decidir, o los marcó como contaminados) no se usan: tienen preferencia desconocida y no se convierten en 0.

Separamos 80 % para entrenar y 20 % para probar, con `random_state=42` y `stratify=interes`. La división se hace **una sola vez** y se reutiliza en los dos clasificadores. Los libros de prueba no se tocan hasta el paso 6. El resultado se guarda en `grupo7_preferencias.csv` en la columna `conjunto`.

In [30]:
ARCH_PREF = ENTREGA + "grupo7_preferencias.csv"

prefs = pd.read_csv(ARCH_PREF, keep_default_na=False)
prefs["interes"] = pd.to_numeric(prefs["interes"], errors="coerce")      # el vacio pasa a NaN
validas = prefs.dropna(subset=["interes"]).copy()
validas["interes"] = validas["interes"].astype(int)

entrenamiento, prueba_pref = train_test_split(
    validas, test_size=0.2, random_state=SEMILLA, stratify=validas["interes"])

# guardamos a cual conjunto pertenece cada libro (las abstenciones quedan con conjunto vacio)
asignacion = {s: "entrenamiento" for s in entrenamiento["slug"]}
asignacion.update({s: "prueba" for s in prueba_pref["slug"]})
prefs["conjunto"] = prefs["slug"].map(asignacion).fillna("")
prefs["interes"] = prefs["interes"].astype("Int64")                      # 1, 0 o vacio, sin decimales
prefs[["slug", "title", "synopsis", "interes", "conjunto"]].to_csv(ARCH_PREF, index=False)

assert not set(entrenamiento["slug"]) & set(prueba_pref["slug"])         # ningun libro en los dos conjuntos
resumen_div = pd.DataFrame({
    "libros": [len(entrenamiento), len(prueba_pref)],
    "interes=1": [int((entrenamiento["interes"] == 1).sum()), int((prueba_pref["interes"] == 1).sum())],
    "interes=0": [int((entrenamiento["interes"] == 0).sum()), int((prueba_pref["interes"] == 0).sum())],
}, index=["entrenamiento", "prueba"])
display(resumen_div)
print("Sin valorar (no se usan):", int(prefs["interes"].isna().sum()))

,libros,interes=1,interes=0
entrenamiento,80,37,43
prueba,20,9,11


Sin valorar (no se usan): 1


# Paso 5. Entrenar dos clasificadores

La entrada es la sinopsis y lo que se aprende es `interes`. Los dos modelos usan la misma regresión logística, con `C=1`, `max_iter=1000` y `class_weight="balanced"`, y esa configuración no se toca en todo el trabajo: no buscamos hiperparámetros.

`class_weight="balanced"` le da más peso a los errores en la clase menos frecuente. Sin eso, con pocos positivos, la regresión suele responder 0 para todos los libros de prueba, porque es lo que más reduce el error. Es parte de la configuración fija y no un ajuste para mejorar resultados.

## Modelo A: TF-IDF

`fit_transform` solo con las sinopsis de entrenamiento y `transform` con las de prueba, así el vocabulario y los pesos salen únicamente del entrenamiento.

In [31]:
def nueva_regresion():
    return LogisticRegression(C=1, max_iter=1000, class_weight="balanced")

txt_ent, y_ent = entrenamiento["synopsis"].tolist(), entrenamiento["interes"].values
txt_pru, y_pru = prueba_pref["synopsis"].tolist(), prueba_pref["interes"].values

vec = TfidfVectorizer()                      # configuracion por defecto
Xa_ent = vec.fit_transform(txt_ent)          # aprende el vocabulario con entrenamiento
Xa_pru = vec.transform(txt_pru)              # solo transforma

modelo_a = nueva_regresion().fit(Xa_ent, y_ent)
pred_a = modelo_a.predict(Xa_pru)            # una lista de 0 y 1, un valor por libro de prueba

print("Matriz TF-IDF de entrenamiento:", Xa_ent.shape, "| de prueba:", Xa_pru.shape)
print("Predicciones del modelo A:", pred_a.tolist())

Matriz TF-IDF de entrenamiento: (80, 4008) | de prueba: (20, 4008)
Predicciones del modelo A: [0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1]


## Modelo B: embeddings

Reutilizamos el modelo multilingüe del TP2 (`distiluse-base-multilingual-cased-v1`, 512 dimensiones). Le pasamos el texto natural, sin limpiar, y queda fijo: lo único que se entrena es la regresión logística que recibe los vectores. El modelo lee como máximo 128 tokens, así que las sinopsis largas se cortan, igual que en el TP2.

In [32]:
if "modelo_emb" not in globals():
    modelo_emb = SentenceTransformer("distiluse-base-multilingual-cased-v1")

Xb_ent = modelo_emb.encode(txt_ent, show_progress_bar=False)
Xb_pru = modelo_emb.encode(txt_pru, show_progress_bar=False)

modelo_b = nueva_regresion().fit(Xb_ent, y_ent)
pred_b = modelo_b.predict(Xb_pru)

print("Matriz de embeddings de entrenamiento:", Xb_ent.shape, "| de prueba:", Xb_pru.shape)
print("Predicciones del modelo B:", pred_b.tolist())

Matriz de embeddings de entrenamiento: (80, 512) | de prueba: (20, 512)
Predicciones del modelo B: [1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1]


# Paso 6. Comprobar las predicciones

Comparamos las predicciones con las respuestas reales de los libros de prueba. Para la clase 1 (el libro le interesa al lector):

- **Precisión**: de los libros que predijimos de interés, cuántos le interesaban.
- **Recall**: de los libros que le interesaban, cuántos detectamos.
- **F1**: el balance entre las dos.

Si un modelo no predice ningún positivo, la precisión no se puede calcular; usamos `zero_division=0` y lo aclaramos.

In [33]:
predicciones = {"A (TF-IDF)": pred_a, "B (embeddings)": pred_b}

for nombre, pred in predicciones.items():
    print("=" * 60, "\nModelo", nombre)
    print(classification_report(y_pru, pred, labels=[0, 1], target_names=["interes=0", "interes=1"], zero_division=0))
    print("Matriz de confusion (filas: real 0 y 1; columnas: predicho 0 y 1)")
    print(confusion_matrix(y_pru, pred, labels=[0, 1]))
    if pred.sum() == 0:
        print("Aviso: este modelo no predijo ningun positivo, por eso la precision de la clase 1 queda en 0.")

Modelo A (TF-IDF)
              precision    recall  f1-score   support

   interes=0       0.50      0.55      0.52        11
   interes=1       0.38      0.33      0.35         9

    accuracy                           0.45        20
   macro avg       0.44      0.44      0.44        20
weighted avg       0.44      0.45      0.45        20

Matriz de confusion (filas: real 0 y 1; columnas: predicho 0 y 1)
[[6 5]
 [6 3]]
Modelo B (embeddings)
              precision    recall  f1-score   support

   interes=0       0.50      0.27      0.35        11
   interes=1       0.43      0.67      0.52         9

    accuracy                           0.45        20
   macro avg       0.46      0.47      0.44        20
weighted avg       0.47      0.45      0.43        20

Matriz de confusion (filas: real 0 y 1; columnas: predicho 0 y 1)
[[3 8]
 [3 6]]


In [34]:
filas = []
for nombre, pred in predicciones.items():
    vn, fp, fn, vp = confusion_matrix(y_pru, pred, labels=[0, 1]).ravel()
    filas.append({"modelo": nombre,
                  "precision": precision_score(y_pru, pred, pos_label=1, zero_division=0),
                  "recall": recall_score(y_pru, pred, pos_label=1, zero_division=0),
                  "F1": f1_score(y_pru, pred, pos_label=1, zero_division=0),
                  "aciertos": int(vp + vn), "errores": int(fp + fn),
                  "falsos positivos": int(fp), "falsos negativos": int(fn)})
tabla_6 = pd.DataFrame(filas).set_index("modelo").round(2)
display(tabla_6)
print(f"Libros de prueba: {len(y_pru)} (positivos: {int(y_pru.sum())}). Un solo libro cambia mucho estas cifras.")

,precision,recall,F1,aciertos,errores,falsos positivos,falsos negativos
modelo,,,,,,,
A (TF-IDF),0.38,0.33,0.35,9,11,5,6
B (embeddings),0.43,0.67,0.52,9,11,8,3


Libros de prueba: 20 (positivos: 9). Un solo libro cambia mucho estas cifras.


Con tan pocos libros de prueba las métricas son orientativas. Para ver cuánto varían repetimos la medición con validación cruzada (`RepeatedStratifiedKFold`, 5 particiones, 10 repeticiones) **solo sobre los libros de entrenamiento** y con la misma configuración. En el modelo A usamos un `Pipeline` para que el `TfidfVectorizer` se ajuste dentro de cada partición. La validación cruzada no se usa para elegir parámetros.

In [35]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEMILLA)
f1_clase1 = make_scorer(f1_score, pos_label=1, zero_division=0)

pipe_a = Pipeline([("tfidf", TfidfVectorizer()), ("logreg", nueva_regresion())])
cv_a = cross_val_score(pipe_a, txt_ent, y_ent, cv=cv, scoring=f1_clase1)
cv_b = cross_val_score(nueva_regresion(), Xb_ent, y_ent, cv=cv, scoring=f1_clase1)

tabla_cv = pd.DataFrame({"F1 clase 1 (media)": [cv_a.mean(), cv_b.mean()],
                         "desvio": [cv_a.std(), cv_b.std()],
                         "particiones": [len(cv_a), len(cv_b)]},
                        index=["A (TF-IDF)", "B (embeddings)"]).round(3)
display(tabla_cv)

,F1 clase 1 (media),desvio,particiones
A (TF-IDF),0.420,0.186,50
B (embeddings),0.533,0.158,50


Ahora miramos dos libros de prueba en detalle: un error del modelo B si lo hubo (si no, uno del modelo A, y si ninguno se equivocó, un acierto) y un acierto. Para el modelo A mostramos las palabras que más empujaron la probabilidad hacia cada lado.

In [36]:
probs_a = modelo_a.predict_proba(Xa_pru)[:, list(modelo_a.classes_).index(1)]
probs_b = modelo_b.predict_proba(Xb_pru)[:, list(modelo_b.classes_).index(1)]

nombres_vocab = vec.get_feature_names_out()
def palabras_clave(texto, n=5):
    contribucion = vec.transform([texto]).multiply(modelo_a.coef_[0]).toarray()[0]
    orden_c = np.argsort(contribucion)
    return list(nombres_vocab[orden_c[::-1][:n]]), list(nombres_vocab[orden_c[:n]])

errores_b = np.where(pred_b != y_pru)[0]
errores_a = np.where(pred_a != y_pru)[0]
aciertos = np.where((pred_a == y_pru) & (pred_b == y_pru))[0]
elegidos = []
if len(errores_b): elegidos.append(errores_b[0])
elif len(errores_a): elegidos.append(errores_a[0])
if len(aciertos): elegidos.append(aciertos[0])
elegidos = (elegidos + [i for i in range(len(y_pru)) if i not in elegidos])[:2]

for i in elegidos:
    libro = prueba_pref.iloc[i]
    a_favor, en_contra = palabras_clave(libro["synopsis"])
    print("=" * 70)
    print(libro["title"], "| respuesta real del lector:", int(libro["interes"]))
    print(libro["synopsis"][:500], "...\n")
    print(f"Modelo A: predijo {pred_a[i]} (prob. interes=1: {probs_a[i]:.2f}) | palabras a favor: {a_favor} | en contra: {en_contra}")
    print(f"Modelo B: predijo {pred_b[i]} (prob. interes=1: {probs_b[i]:.2f})")

La promesa del barón | respuesta real del lector: 0
Annalise Berry está a punto de ser presentada en sociedad. Sin embargo, la joven no se engaña, proviene de una familia sencilla en Escocia, y aunque su dote es sustanciosa, sabe que no será suficiente para lograr el final de cuento de hadas que su protector desea para ella. El futuro matrimonio de Jacob Peasly, barón Peasly, está pactado desde su niñez. Debe desposar a la hija de un hombre poderoso, que no lo dejará olvidar la promesa hecha. Pero, desde que conoció a Annalise, su corazón lo empu ...

Modelo A: predijo 1 (prob. interes=1: 0.53) | palabras a favor: ['su', 'desde', 'para', 'familia', 'de'] | en contra: ['la', 'hace', 'hombre', 'primer', 'le']
Modelo B: predijo 1 (prob. interes=1: 0.50)
El libro de los universos | respuesta real del lector: 1
Acostumbrados como estamos a pensar que el Universo es único, centramos nuestros esfuerzos en averiguar qué contiene y cuál es su estructura. En El libro de los universos el distingu

**Qué pasó con cada libro.**

*La promesa del barón* es un falso positivo de los dos modelos. El lector lo había valorado con 0 (es una novela romántica de época) y los dos lo marcaron con 1, aunque apenas: 0,53 el modelo A y 0,50 el B, justo en el límite. Las palabras que más empujaron al modelo A hacia el 1 son «su», «desde», «para», «familia» y «de». Salvo «familia», son palabras vacías. `TfidfVectorizer` está con su configuración por defecto, sin lista de stopwords, y con solo 80 libros el modelo termina dándoles peso a palabras que aparecen en cualquier sinopsis.

*El libro de los universos* es un acierto de los dos, porque el lector lo había valorado con 1. Acá las palabras a favor sí dicen algo del tema: «barrow» (el autor) y «física». Es un libro de divulgación científica, y en el paso 8 se ve que la ciencia es uno de los temas que más se repiten entre los libros que le interesan. Igual las probabilidades (0,54 y 0,56) siguen cerca de 0,5: ningún modelo está muy seguro.

# Paso 7. Convertir las predicciones en recomendaciones

Para recomendar hace falta ordenar, no solo decir 0 o 1. Con `predict_proba` obtenemos la probabilidad estimada de `interes=1` y revisamos en `classes_` qué columna es esa clase. Armamos tres listas sobre **exactamente los mismos libros de prueba**:

1. Clasificador con TF-IDF.
2. Clasificador con embeddings.
3. Similitud con el perfil del lector: el promedio de los embeddings de los libros con `interes=1` **de entrenamiento**, comparado con cada libro de prueba por coseno (la idea del TP2).

En cada lista tomamos los cinco primeros y calculamos precision@5: cuántos tenían `interes=1`, dividido 5.

In [37]:
col1_a, col1_b = list(modelo_a.classes_).index(1), list(modelo_b.classes_).index(1)
assert modelo_a.classes_[col1_a] == 1 and modelo_b.classes_[col1_b] == 1
puntaje_a = modelo_a.predict_proba(Xa_pru)[:, col1_a]
puntaje_b = modelo_b.predict_proba(Xb_pru)[:, col1_b]

perfil = Xb_ent[y_ent == 1].mean(axis=0, keepdims=True)      # solo positivos de entrenamiento
puntaje_perfil = cosine_similarity(perfil, Xb_pru)[0]

K = 5
def primeros(puntaje):
    return np.argsort(-puntaje, kind="stable")[:K]            # de mayor a menor; empates por orden de aparicion

listas = {"TF-IDF + clasificador": puntaje_a,
          "Embeddings + clasificador": puntaje_b,
          "Similitud con el perfil": puntaje_perfil}

tabla_7 = []
for nombre, puntaje in listas.items():
    top = primeros(puntaje)
    aciertos_k = int((y_pru[top] == 1).sum())
    tabla_7.append({"lista": nombre, "aciertos en los 5 primeros": aciertos_k, "precision@5": aciertos_k / K})
    print("=" * 70, "\n" + nombre)
    display(pd.DataFrame({"titulo": prueba_pref["title"].values[top],
                          "puntaje": puntaje[top].round(3),
                          "interes real": y_pru[top]}))

azar = y_pru.mean()
maximo = min(K, int(y_pru.sum())) / K
display(pd.DataFrame(tabla_7).set_index("lista"))
print(f"Referencia al azar (proporcion de 1 en prueba): {azar:.2f}")
print(f"Maximo alcanzable con {int(y_pru.sum())} positivos en prueba: {maximo:.2f}")

TF-IDF + clasificador


,titulo,puntaje,interes real
0,Viaje a Yucatán,0.538,0
1,El libro de los universos,0.536,1
2,La promesa del barón,0.531,0
3,Cinco destinos oscuros,0.523,0
4,Nación TV,0.517,0


Embeddings + clasificador


,titulo,puntaje,interes real
0,Los narcisos: han tomado el poder,0.584,0
1,El médico de Tesalia,0.563,1
2,Fantasías eróticas,0.560,1
3,El libro de los universos,0.558,1
4,El frente del Este,0.558,1


Similitud con el perfil


,titulo,puntaje,interes real
0,Algo va mal,0.549,0
1,Los narcisos: han tomado el poder,0.520,0
2,Recuerdos de mi vida,0.504,0
3,"Ulises, un arquetipo de la existencia humana",0.476,1
4,Matriarcadia,0.465,0


,aciertos en los 5 primeros,precision@5
lista,,
TF-IDF + clasificador,1,0.2
Embeddings + clasificador,4,0.8
Similitud con el perfil,1,0.2


Referencia al azar (proporcion de 1 en prueba): 0.45
Maximo alcanzable con 9 positivos en prueba: 1.00


**¿Los clasificadores recomendaron mejor que la similitud?** Uno sí. El clasificador con embeddings puso 4 libros de interés entre sus 5 primeros (precision@5 de 0,80), mientras que TF-IDF con clasificador y la similitud con el perfil acertaron 1 de 5 (0,20). Elegir al azar daría 0,45 en promedio, así que esas dos listas quedaron por debajo del azar y la de embeddings bastante por encima.

Con 5 libros por lista hay que tomarlo con cuidado. Si sacamos 5 libros al azar de los 20 de prueba (9 positivos), la probabilidad de acertar 4 o más es de casi 10 %, o sea que pudo haber algo de suerte. Lo que sí coincide es que en la validación cruzada del paso 6 también rindió mejor embeddings (F1 0,53 contra 0,42).

La similitud con el perfil fue la que peor anduvo. El perfil es el promedio de los 37 libros de entrenamiento que le interesaron al lector, y son muy variados (ciencia, historia, novelas). Creemos que el promedio termina en un punto medio que no se parece a ninguno en particular. El clasificador, en cambio, aprende también de los 43 libros que no le interesaron, y eso le sirve para separar.

# Paso 8. Recomendar libros del catálogo completo

Usamos el clasificador con embeddings del paso 5, **sin volver a ajustarlo**, sobre los libros del catálogo limpio que el lector no valoró. Excluimos:

- todos los libros de `grupo7_preferencias.csv`, incluidas las abstenciones;
- los libros que quedaron en `grupo7_sospechosos.csv`, porque el lector pudo agregar algunos con motivo `revision_manual` después de que se guardó `catalogo_limpio.csv`;
- cualquier otra versión de un libro ya valorado (mismo autor y título muy parecido), con el criterio del paso 1.

In [38]:
limpio = pd.read_csv(RUTA + "catalogo_limpio.csv", keep_default_na=False)
sospechosos_final = pd.read_csv(ARCH_SOSP)

valorados = limpio[limpio["slug"].isin(prefs["slug"])]
pool = limpio[~limpio["slug"].isin(prefs["slug"]) & ~limpio["slug"].isin(sospechosos_final["slug"])].reset_index(drop=True)
print("Catalogo limpio:", len(limpio), "| valorados:", len(valorados), "| candidatos:", len(pool))

# embeddings de todos los candidatos (con CPU tarda unos minutos)
Xpool = modelo_emb.encode(pool["synopsis"].tolist(), batch_size=64, show_progress_bar=True)
pool["puntuacion"] = modelo_b.predict_proba(Xpool)[:, col1_b]
ranking = pool.sort_values("puntuacion", ascending=False, kind="stable").reset_index(drop=True)

def es_version_de_valorado(libro):
    """Mismo autor, titulo clave muy parecido y mismos numeros de tomo que un libro ya valorado."""
    k, n = clave_titulo(libro.title), numeros_titulo(libro.title)
    for v in valorados[valorados["autores"] == libro.autores].itertuples():
        if numeros_titulo(v.title) == n and jellyfish.jaro_winkler_similarity(k, clave_titulo(v.title)) >= UMBRAL_TITULO:
            return True
    return False

print("Distribucion de la puntuacion (candidatos):")
display(ranking["puntuacion"].describe().round(3).to_frame().T)

Catalogo limpio: 7890 | valorados: 101 | candidatos: 7789


Batches:   0%|          | 0/122 [00:00<?, ?it/s]

Distribucion de la puntuacion (candidatos):


,count,mean,std,min,25%,50%,75%,max
puntuacion,7789.0,0.505,0.06,0.314,0.463,0.504,0.548,0.699


## Resumir las recomendaciones

Una sinopsis completa es larga para una lista. Para cada libro hacemos un **resumen extractivo de dos oraciones**: dividimos la sinopsis en oraciones, las representamos con TF-IDF, calculamos la similitud del coseno entre ellas y nos quedamos con las dos más centrales (las que más se parecen al resto), en su orden original.

In [39]:
def resumen_extractivo(sinopsis, n_oraciones=2):
    frases = oraciones(sinopsis)
    if len(frases) <= n_oraciones:
        return " ".join(frases) if frases else sinopsis
    X = TfidfVectorizer().fit_transform(frases)
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)                                  # una oracion no cuenta contra si misma
    centrales = np.argsort(-sim.sum(axis=1), kind="stable")[:n_oraciones]
    return " ".join(frases[i] for i in sorted(centrales))   # en el orden original

def recomendar(n=5):
    """Devuelve identificador, titulo, puntuacion y resumen de los n libros recomendados."""
    salida = []
    for r in ranking.itertuples():
        if es_version_de_valorado(r):
            continue
        salida.append({"slug": r.slug, "title": r.title, "puntuacion": round(r.puntuacion, 3),
                       "resumen": resumen_extractivo(r.synopsis)})
        if len(salida) == n:
            break
    return pd.DataFrame(salida)

top5 = recomendar(5)
for r in top5.itertuples():
    print(f"{r.title}  (slug: {r.slug}, puntuacion: {r.puntuacion})")
    print("   ", r.resumen, "\n")

Asamblea mortal  (slug: asamblea-mortal, puntuacion: 0.699)
    Un consejero del Servicio Secreto Británico ha tenido una idea para cazar al siniestro «Naif», que se dedica a vender información sobre agentes secretos de todos los servicios para que puedan ser asesinados. Cada uno deberá enviar a su castillo al mejor espía del que dispongan para una asamblea en la que expondrá su plan para la captura del espía mercenario. 

Hitler y las teorías de la conspiración  (slug: hitler-y-las-teorias-de-la-conspiracion, puntuacion: 0.689)
    No obstante, en la actualidad estamos viviendo un auge de las teorías de la conspiración y, entre ellas, los relatos revisionistas de la historia del Tercer Reich se han vuelto más populares que ningún otro tema a causa de supuestas evidencias recién descubiertas y nuevos ángulos de investigación. Richard Evans, reconocido historiador del Tercer Reich, elige cinco de las teorías conspirativas más duraderas sobre el período nazi y las analiza con rigor: la s

**Un resumen que funcionó y uno que perdió información.**

Funcionó bien el de *Asamblea mortal*. La sinopsis tiene tres oraciones y el método dejó la primera (quién arma el plan y a quién quiere cazar) y la tercera (la asamblea de espías en el castillo). Se perdió la del medio, que dice que se unen varios servicios secretos, pero eso se deduce de la tercera.

Perdió información importante el de *El hombre en la red*. De cuatro oraciones quedaron «Su mujer desaparece» y la última, la de que tiene que demostrar su inocencia. Se cayeron el nombre del protagonista y la oración que explica el conflicto: que las sospechas recaen sobre él. Sin eso no se entiende de qué tiene que demostrar su inocencia. En *Resurrección* pasó algo parecido, porque se perdió cómo fueron los asesinatos, que es el gancho del libro. En *El espía de Madrid* el resumen es la sinopsis completa, que tiene solo dos oraciones.

Una explicación posible es que el TF-IDF de las oraciones también cuenta palabras vacías («su», «de», «contra»). Con tan pocas oraciones, la centralidad depende mucho de esas palabras compartidas y poco del contenido.

**Comentario del lector sobre las cinco recomendaciones** (Valentino Civetta; no entra en las métricas de prueba):

> De las cinco me interesan *Hitler y las teorías de la conspiración* y *El espía de Madrid. Barcelona, 1936*. No me interesan *Asamblea mortal*, *Resurrección* ni *El hombre en la red*.

Le interesaron 2 de 5. Los dos que eligió tienen un fondo histórico real (el nazismo y la Guerra Civil española), y los tres que descartó son policiales o de espías de pura ficción. Parece que el modelo aprendió algo parecido a «intriga, guerra y conspiraciones», sin distinguir entre la historia real y la ficción de género.

## Describir el perfil del lector

Con spaCy extraemos las frases sustantivas (`noun_chunks`) y las entidades nombradas de las sinopsis de **entrenamiento** con `interes=1` y con `interes=0`, y mostramos las diez más frecuentes de cada grupo. Para agrupar variantes ("la guerra", "las guerras") usamos el lema del núcleo de cada frase. Dejamos afuera los núcleos que no son sustantivos y las palabras vacías.

In [40]:
if "nlp" not in globals():
    nlp = spacy.load("es_core_news_lg")

def perfil_de(textos, top=10):
    frases, entidades_ = collections.Counter(), collections.Counter()
    for doc in nlp.pipe(textos, batch_size=32):
        for chunk in doc.noun_chunks:
            raiz = chunk.root
            if raiz.pos_ in ("NOUN", "PROPN") and not raiz.is_stop:
                frases[raiz.lemma_.lower()] += 1
        for ent in doc.ents:
            entidades_[(ent.text.strip(), ent.label_)] += 1
    return frases.most_common(top), entidades_.most_common(top)

for valor, nombre in ((1, "interes=1 (le interesan)"), (0, "interes=0 (no le interesan)")):
    textos = entrenamiento.loc[entrenamiento["interes"] == valor, "synopsis"].tolist()
    frases_top, ents_top = perfil_de(textos)
    print("=" * 70, f"\n{nombre}: {len(textos)} sinopsis")
    display(pd.DataFrame(frases_top, columns=["nucleo (lema)", "veces"]))
    display(pd.DataFrame([(t, l, c) for (t, l), c in ents_top], columns=["entidad", "tipo", "veces"]))

interes=1 (le interesan): 37 sinopsis


,nucleo (lema),veces
0,libro,20
1,historia,11
2,ciencia,10
3,vida,8
4,personaje,8
5,guerra,8
6,novela,8
7,año,8
8,teoría,8
9,siglo,8


,entidad,tipo,veces
0,Sara,PER,4
1,Europa,LOC,4
2,Himmler,PER,4
3,Salomón,PER,3
4,J. Volpi,PER,3
5,Mike Hammer,PER,3
6,Alemania,LOC,3
7,Julia,PER,3
8,Roth,PER,2
9,Segunda Guerra Mundial,MISC,2


interes=0 (no le interesan): 43 sinopsis


,nucleo (lema),veces
0,libro,33
1,vida,23
2,historia,22
3,obra,19
4,autor,16
5,mundo,13
6,gato,13
7,año,11
8,país,9
9,hombre,8


,entidad,tipo,veces
0,Chesterton,PER,5
1,Roma,LOC,5
2,París,LOC,4
3,España,LOC,4
4,Kamanita,LOC,3
5,El libro,MISC,3
6,Barcelona,LOC,3
7,PDF,MISC,3
8,Rusia,LOC,3
9,Orson Welles,PER,3


**¿Esta lista describe lo que le gusta al lector?** Según el lector, sí, bastante. Las palabras que separan los dos grupos están en el medio de la lista, porque «libro», «historia», «vida» y «año» aparecen en los dos y no dicen nada. En los libros que le interesaron se repiten «ciencia», «teoría», «guerra», «personaje», «novela» y «siglo», y entidades como Himmler, Alemania, la Segunda Guerra Mundial y el detective Mike Hammer. En los que no, «obra», «autor», «mundo», «gato» y «país», junto con Chesterton, Roma y París. La entidad «PDF» de ese grupo sale del aviso de Lectulandia que recomienda bajar la poesía en EPUB: de los cuatro libros valorados que lo tienen, tres son 0.

La lista coincide con las recomendaciones de arriba, donde el modelo eligió espionaje y Segunda Guerra. Una puntuación alta es una estimación del modelo, no una garantía de que el libro le va a gustar después de leerlo.

# Conclusiones

En la auditoría cada detector encontró un tipo de contaminación y ninguno los encontró todos. Combinados recuperaron los 30 libros contaminados de prueba, con precisión 0,79. Los clasificadores de oraciones fueron los que más aportaron. El detector de idioma fue el más flojo y el que más falsos positivos sumó, y en el catálogo completo eso se nota: 703 libros marcados, más de los que esperábamos.

Para las preferencias, con 80 libros de entrenamiento los dos clasificadores quedaron cerca del azar en los 20 libros de prueba (F1 de la clase 1: 0,35 TF-IDF y 0,52 embeddings), y la validación cruzada mostró mucha variación (0,42 ± 0,19 y 0,53 ± 0,16). Como referencia, un modelo que respondiera que sí a todo tendría F1 0,62 en prueba, así que el F1 solo no alcanza para decir que aprendieron algo. La diferencia apareció al ordenar: el clasificador con embeddings tuvo precision@5 de 0,80, y TF-IDF y la similitud con el perfil, 0,20.

La distancia entre TF-IDF y embeddings fue mucho más grande en las preferencias que en la auditoría. Para detectar spam había más de 2.000 oraciones y patrones literales que se repiten, y TF-IDF los aprende igual o mejor. Para el gusto del lector había 80 sinopsis y un vocabulario de 4.008 palabras sacado de esos pocos textos. Ahí ayuda que los embeddings traigan el significado aprendido de antes.

El límite principal del experimento es el tamaño: un solo lector, 100 valoraciones y 20 libros de prueba, donde un libro cambia 0,05 la exactitud y todavía más la precisión o el recall. Además SBERT lee solo los primeros 128 tokens de cada sinopsis. En la revisión final, el lector aceptó 2 de las 5 recomendaciones.